# 課程 08 - 多代理設計模式


## 設定


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 點解要用多智能體系統？

現實世界嘅任務，例如旅遊規劃，涉及好多唔同嘅專業知識——物流、本地資訊、預算等等。單一智能體試圖處理晒所有嘢好快就會變得難以掌握。

多智能體系統透過<strong>專業化</strong>嚟解決呢個問題：每個智能體專注於一個專業範疇，產出嘅結果比起泛泛而談嘅智能體更高質素。佢哋亦提升咗<strong>可擴展性</strong>——你可以加入新智能體（例如航班專家、餐廳評論員），而唔使重寫現有嘅流程。呢啲智能體透過有結構嘅管道組合埋一齊，喺佢哋之間傳遞上下文。


## 建立專門代理人


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 建立一個順序工作流程

`WorkflowBuilder` 讓你將代理串接成一個有向圖。這裡我們建立一個簡單的兩步驟管線：**TravelPlanner** 草擬行程，然後 **TravelConcierge** 審查並優化它。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 為工作流程加入更多代理人

多代理人模式最大好處之一就是擴充十分容易。以下我們加入一個 **BudgetReviewer** 代理人，負責檢查計劃是否符合旅客的預算，標示可能會令成本超出限制的項目，並建議節省開支的替代方案。工作流程現在依次執行三個代理人：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 摘要

在本課程中，您學會了如何：

1. <strong>建立專門的代理人</strong> — 每個代理人都有專注的角色（規劃、禮賓、預算審核）。
2. **使用 `WorkflowBuilder` 和 `add_edge` 將代理人連接成序列工作流程**。
3. <strong>從多代理人管道中串流輸出</strong>，追蹤正在發言的代理人。
4. <strong>透過新增代理人擴充工作流程</strong>，而不需修改現有代理人。

多代理人設計模式讓每個代理人保持簡單，同時產出比任何單一代理人獨自達成更豐富、更完整審核的結果。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
